In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from catboost import CatBoostRegressor
from typing import Dict, Any, Optional, List, Tuple
import warnings 
warnings.filterwarnings("ignore")

%matplotlib inline 

## Table of Contents
### Feature Description
### Baseline Model
### Baseline Model lag features
### Conclusions

# Feature Description

In [ ]:
data = pd.read_csv('/Users/inji/mllearn/data/01_raw/bike_sharing_hour_data.csv')
data.head(3)


In [ ]:
data.dtypes

## Step 1 — Prepare the table (no lags yet)

Load **renamed** data (Kedro output), build a datetime, sort by time, drop leakage columns.

- `casual_users` + `registered_users` = `total_users` → do not use them as features
- `instant` is just a row id
- We parse `dteday` + `hr` so a later time split is valid

In [ ]:
# 1. Load the Kedro-renamed table (not the raw file)
df = pd.read_csv("../data/02_intermediate/renamed_data.csv")
df.dtypes

In [ ]:
# 1. Load the Kedro-renamed table (not the raw file)
df = pd.read_csv("../data/02_intermediate/renamed_data.csv")

# 2. dteday is a string + hr is 0–23. Combine into a real timestamp.
df["dteday"] = pd.to_datetime(df["dteday"])
df["datetime"] = df["dteday"] + pd.to_timedelta(df["hr"], unit="h")

# 3. Time order matters for any later split (even without lags).
df = df.sort_values("datetime").reset_index(drop=True)

# 4. Leakage / IDs: never put these in X.
LEAKAGE_OR_ID = ["instant", "casual_users", "registered_users"]
df_model = df.drop(columns=LEAKAGE_OR_ID)


# 5. Features vs target. datetime is for splitting/plotting, not a model feature.
TARGET = "total_users"
feature_cols = [
    c
    for c in df_model.columns
    if c not in {TARGET, "datetime", "dteday"}
]
X = df_model[feature_cols]
y = df_model[TARGET]

print(df["datetime"].min(), "→", df["datetime"].max())
print("rows:", len(df_model))
print("features:", feature_cols)
print(
    "check leakage (should be 0):",
    (df["casual_users"] + df["registered_users"] - df["total_users"]).abs().sum(),
)
X.head(3)


## Step 2 — Time-based train/test split

Rows are already sorted by time (Step 1): oldest at the top, newest at the bottom.

Count how many rows are **before** `2012-10-01`. Take that many from the **start** as train; the rest as test.

`iloc[:n]` means “first n rows”. `iloc[n:]` means “from row n to the end”.

In [ ]:
df_model.groupby("dteday")['dteday'].value_counts()

In [ ]:
# Rows are already in time order (Step 1).
CUTOFF = pd.Timestamp("2012-10-01")
n_train = (df_model["datetime"] < CUTOFF).sum()  # how many rows are in the past

# First n_train rows = train; remaining rows = test
X_train = X.iloc[:n_train]
y_train = y.iloc[:n_train]
X_test = X.iloc[n_train:]
y_test = y.iloc[n_train:]

print("n_train:", n_train, "n_test:", len(X_test))
print("last train time:", df_model["datetime"].iloc[n_train - 1])
print("first test time:", df_model["datetime"].iloc[n_train])

## Step 3 — Fit a simple model (no lags)

Train only on `X_train` / `y_train`. Score on `X_test` / `y_test`.

Compare two predictions:
- **Mean baseline:** always guess the average `total_users` from train (sanity check)
- **HistGradientBoostingRegressor:** sklearn tree model using calendar + weather columns

MAE = average absolute error in rider counts. RMSE penalizes large misses more.

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# 1) Dummy baseline: always predict the training-set mean
y_mean = y_train.mean()
pred_mean = np.full(len(y_test), y_mean)

# 2) Simple tree model — no lags, just the feature columns in X
model = HistGradientBoostingRegressor(random_state=42)
model.fit(X_train, y_train)
pred = model.predict(X_test)

def score(name, y_true, y_hat):
    mae = mean_absolute_error(y_true, y_hat)
    rmse = mean_squared_error(y_true, y_hat) ** 0.5
    print(f"{name:20s}  MAE={mae:6.1f}  RMSE={rmse:6.1f}")

score("train mean", y_test, pred_mean)
score("HistGradientBoosting", y_test, pred)

print("train mean total_users:", round(y_mean, 1))

## Step 4 — Where is the error?

Plot **actual vs predicted** for the first test week, then **mean |error| by hour of day** on the whole test set.

If commute hours (8, 17–18) have the largest MAE, the model has the daily shape but misses peak size.

In [ ]:
# Align test timestamps with predictions (same row order as X_test / y_test)
test = pd.DataFrame({
    "datetime": df_model["datetime"].iloc[n_train:].to_numpy(),
    "hr": X_test["hr"].to_numpy(),
    "actual": y_test.to_numpy(),
    "pred": pred,
})
test["abs_err"] = (test["actual"] - test["pred"]).abs()

week = test[(test["datetime"] >= "2012-10-01") & (test["datetime"] < "2012-10-08")]

fig, axes = plt.subplots(2, 1, figsize=(10, 8))

axes[0].plot(week["datetime"], week["actual"], label="actual")
axes[0].plot(week["datetime"], week["pred"], label="predicted")
axes[0].set_title("Test week 1–7 Oct 2012: actual vs predicted total_users")
axes[0].set_ylabel("riders per hour")
axes[0].legend()

err_by_hr = test.groupby("hr")["abs_err"].mean()
axes[1].bar(err_by_hr.index, err_by_hr.values)
axes[1].set_title("Test set: mean |error| by hour of day")
axes[1].set_xlabel("hr")
axes[1].set_ylabel("MAE (riders)")
axes[1].set_xticks(range(24))

plt.tight_layout()
plt.show()

print(err_by_hr.sort_values(ascending=False).head(5))

### Features lagged - Model to be used HistGradientBoostingRegressor

In [ ]:
df.head(5)

In [ ]:
def  lag_features(df:pd.DataFrame, lags:list[int])->pd.DataFrame:
    df_out = df.copy()
   
    for lag in lags:
        df_out[f"datetime_lag_{lag}_hr"] = df_out["datetime"] + pd.Timedelta(days=lag/24)
        df_out_few = df_out[[f"datetime_lag_{lag}_hr","total_users"]].rename(columns={"total_users":f"total_users_lag_{lag}_hr"})
        df_new =pd.merge(df, df_out_few, how = "left", left_on = "datetime", right_on = f"datetime_lag_{lag}_hr")
    return df_new.drop(columns=[f"datetime_lag_{lag}_hr"])

## lag_features(df,[24,168]).head(48)
df_lag = lag_features(df,[24,168])


In [ ]:
# columns to be removed 

columns_to_remove = ["registered_users", "casual_users", "instant", "dteday", "datetime"]

feature_cols = [cols for cols in df_lag.columns if cols not in columns_to_remove + [TARGET]]
X=df_lag[feature_cols]
y=df_lag[TARGET]

print("Time range in df model lagged",df_lag["datetime"].min(),"→",df_lag["datetime"].max())
print("rows:",len(df_lag))
print("features in lag model:",feature_cols,"\n")



# Removing the columns like registered_users, casual_users, instant, dteday, datetime 


In [ ]:
# Split Test and Train 
CUTOFF = pd.Timestamp("2012-10-01") 
n_train = (df_lag['datetime']< CUTOFF).sum()

X_train = X.iloc[:n_train]
Y_train =y.iloc[:n_train]
X_test = X.iloc[n_train:]
Y_test =y.iloc[n_train:]

print("n_train:", n_train, "n_test:", len(X_test))
print("last train time:", df_lag["datetime"].iloc[n_train - 1])


In [ ]:
#Dummy baseline
y_mean = Y_train.mean()
pred_mean = np.full(len(Y_test), y_mean)

# Model Fitting
model = HistGradientBoostingRegressor(random_state=42)
model.fit(X_train, Y_train)
pred = model.predict(X_test)

# Model Evaluation
score("train mean", Y_test, pred_mean)
score("HistGradientBoosting", Y_test, pred)

print("train mean total_users:", round(y_mean, 1))

# Plotting



## Model comparison (Kedro report)

After `uv run kedro run --pipeline=training`, **`comparison.png`** overlays actual vs predicted-HG / CB / RF (same colors on the error bars). This does **not** retrain in the notebook.


In [ ]:
import json
from IPython.display import Image, display, Markdown
from pathlib import Path

report = Path("../data/08_reporting")
comparison = json.loads((report / "comparison.json").read_text())

for name, scores in comparison.items():
    print(f"{name:16s}  MAE={scores['mae']:6.1f}  RMSE={scores['rmse']:6.1f}")

display(Image(filename=str(report / "comparison.png")))


In [ ]:
# comparison.png is the side-by-side grid (charts + scores).
